# Pulso — Fase 2 (Ir Além): Classificação de ECG com Rede Neural MLP

**Projeto:** Pulso — Inteligência Cardiológica com Consciência de Dados  
**Autor:** Richard Schmitz | RM 567951  
**Disciplina:** Inteligência Artificial — FIAP | Turma 2TIAOR-2026  

---

## Objetivo

Este notebook implementa o Ir Além 2 da Fase 2 do projeto Pulso.  
Utilizando o dataset MIT-BIH Arrhythmia (via Kaggle — shayanfazeli/heartbeat),  
o sistema treina uma Rede Neural MLP (Perceptron Multicamadas) para classificar  
sinais de ECG como **normal** ou **anormal**.

### Sobre o dataset

O dataset contém sinais de ECG segmentados em batimentos individuais,  
cada um representado por 187 pontos de amplitude ao longo do tempo.  
As classes originais são:

| Classe | Descrição | Mapeamento |
|--------|-----------|------------|
| 0 | Normal | Normal |
| 1 | Supraventricular | Anormal |
| 2 | Ventricular | Anormal |
| 3 | Fusão | Anormal |
| 4 | Não classificado | Anormal |

Para este projeto, as classes 1–4 são agrupadas em **anormal**,  
criando um problema de classificação binária alinhado com o objetivo clínico:  
identificar se um batimento cardíaco é normal ou requer atenção médica.

### Conexão com a Fase 1

Este módulo complementa o dataset numérico UCI Cleveland da Fase 1.  
Enquanto o Cleveland usa variáveis clínicas para prever doença coronariana,  
este módulo usa o sinal elétrico do coração para detectar arritmias —  
duas perspectivas complementares do mesmo problema cardiovascular.

## 1. Instalação e importações

> **No Google Colab:** execute a célula abaixo para instalar o kaggle e baixar o dataset.

In [ ]:
# Instalar kaggle se necessário
# !pip install kaggle -q

# Para baixar o dataset via Kaggle API:
# 1. Acesse https://www.kaggle.com/settings e gere um token API
# 2. Faça upload do kaggle.json no Colab
# 3. Execute:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d shayanfazeli/heartbeat --unzip

# Alternativa: baixar manualmente de https://www.kaggle.com/datasets/shayanfazeli/heartbeat
# e fazer upload dos arquivos mitbih_train.csv e mitbih_test.csv no Colab

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow versão: {tf.__version__}')
print('Importações concluídas.')

## 2. Carregamento do dataset

In [ ]:
# Ajuste os caminhos conforme onde os arquivos estão no Colab
df_train = pd.read_csv('mitbih_train.csv', header=None)
df_test  = pd.read_csv('mitbih_test.csv',  header=None)

print(f'Treino : {df_train.shape[0]:,} batimentos, {df_train.shape[1]-1} features + 1 rótulo')
print(f'Teste  : {df_test.shape[0]:,} batimentos, {df_test.shape[1]-1} features + 1 rótulo')

print('\nDistribuição original das classes (treino):')
classes_orig = {
    0: 'Normal',
    1: 'Supraventricular',
    2: 'Ventricular',
    3: 'Fusão',
    4: 'Não classificado'
}
contagem = df_train.iloc[:, -1].value_counts().sort_index()
for cls, count in contagem.items():
    print(f'  Classe {int(cls)} ({classes_orig[int(cls)]}): {count:,} batimentos')

## 3. Pré-processamento

### 3.1 Conversão para classificação binária

Classes 1–4 são agrupadas em **anormal (1)**.  
Classe 0 permanece como **normal (0)**.

### 3.2 Balanceamento

O dataset original é fortemente desbalanceado (~72% normal).  
Para evitar que o modelo aprenda a prever sempre "normal",  
aplicamos undersampling na classe majoritária.

In [ ]:
def preparar_dados(df, n_amostras_por_classe=5000):
    """
    Separa features e rótulos, converte para binário e balanceia.
    """
    X = df.iloc[:, :-1].values
    y_orig = df.iloc[:, -1].values.astype(int)

    # Conversão binária: 0 = normal, 1 = anormal
    y = (y_orig > 0).astype(int)

    # Balanceamento por undersampling
    idx_normal  = np.where(y == 0)[0]
    idx_anormal = np.where(y == 1)[0]

    n = min(n_amostras_por_classe, len(idx_normal), len(idx_anormal))

    np.random.seed(42)
    idx_normal_bal  = np.random.choice(idx_normal,  n, replace=False)
    idx_anormal_bal = np.random.choice(idx_anormal, n, replace=False)

    idx_bal = np.concatenate([idx_normal_bal, idx_anormal_bal])
    np.random.shuffle(idx_bal)

    return X[idx_bal], y[idx_bal]


X_train, y_train = preparar_dados(df_train, n_amostras_por_classe=5000)
X_test,  y_test  = preparar_dados(df_test,  n_amostras_por_classe=1000)

print(f'Treino balanceado : {X_train.shape[0]:,} batimentos')
print(f'  Normal  : {(y_train == 0).sum():,}')
print(f'  Anormal : {(y_train == 1).sum():,}')
print(f'\nTeste balanceado  : {X_test.shape[0]:,} batimentos')
print(f'  Normal  : {(y_test == 0).sum():,}')
print(f'  Anormal : {(y_test == 1).sum():,}')

### 3.3 Normalização

Redes neurais convergem mais rápido quando as features têm média zero e desvio padrão unitário.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Normalização concluída.')
print(f'Shape final de treino : {X_train_scaled.shape}')
print(f'Shape final de teste  : {X_test_scaled.shape}')

## 4. Visualização de um batimento cardíaco

Cada linha do dataset representa um batimento cardíaco segmentado em 187 pontos de amplitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

idx_normal  = np.where(y_train == 0)[0][0]
idx_anormal = np.where(y_train == 1)[0][0]

axes[0].plot(X_train[idx_normal], color='steelblue', linewidth=1.5)
axes[0].set_title('Batimento Normal', fontsize=13)
axes[0].set_xlabel('Ponto de tempo')
axes[0].set_ylabel('Amplitude')
axes[0].grid(alpha=0.3)

axes[1].plot(X_train[idx_anormal], color='firebrick', linewidth=1.5)
axes[1].set_title('Batimento Anormal', fontsize=13)
axes[1].set_xlabel('Ponto de tempo')
axes[1].set_ylabel('Amplitude')
axes[1].grid(alpha=0.3)

plt.suptitle('Comparação de Batimentos Cardíacos — Dataset MIT-BIH', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/batimentos_exemplo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em data/batimentos_exemplo.png')

## 5. Arquitetura da rede neural MLP

A rede recebe 187 features (pontos do sinal de ECG) e produz uma probabilidade binária.

```
Input (187)  →  Dense(128, ReLU)  →  Dropout(0.3)
             →  Dense(64, ReLU)   →  Dropout(0.3)
             →  Dense(32, ReLU)
             →  Dense(1, Sigmoid)  →  Output (normal / anormal)
```

- **ReLU**: evita o problema do gradiente desaparecendo em camadas profundas
- **Dropout**: desativa aleatoriamente neurônios durante o treino para reduzir overfitting
- **Sigmoid**: produz uma probabilidade entre 0 e 1 para classificação binária

In [ ]:
def construir_modelo(input_dim):
    modelo = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


modelo = construir_modelo(X_train_scaled.shape[1])
modelo.summary()

## 6. Treinamento

In [ ]:
# EarlyStopping interrompe o treino se a validação parar de melhorar
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

historico = modelo.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTreinamento concluído em {len(historico.history["loss"])} épocas.')

## 7. Curvas de aprendizado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historico.history['accuracy'],     label='Treino',   color='steelblue')
axes[0].plot(historico.history['val_accuracy'], label='Validação', color='firebrick')
axes[0].set_title('Acurácia por época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(historico.history['loss'],     label='Treino',   color='steelblue')
axes[1].plot(historico.history['val_loss'], label='Validação', color='firebrick')
axes[1].set_title('Loss por época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Curvas de Aprendizado — MLP ECG', fontsize=14)
plt.tight_layout()
plt.savefig('../data/curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Avaliação no conjunto de teste

In [ ]:
y_prob = modelo.predict(X_test_scaled).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acuracia = accuracy_score(y_test, y_pred)
print(f'Acurácia no conjunto de teste: {acuracia:.2%}\n')

print('Relatório de classificação:')
print(classification_report(
    y_test, y_pred,
    target_names=['Normal', 'Anormal']
))

print('Matriz de confusão:')
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Real: Normal', 'Real: Anormal'],
    columns=['Previsto: Normal', 'Previsto: Anormal']
)
print(cm_df)

print('\nInterpretação clínica:')
fn = cm[1][0]
fp = cm[0][1]
print(f'  Falsos negativos (anormal classificado como normal): {fn}')
print(f'  Falsos positivos (normal classificado como anormal): {fp}')
print(f'  Em triagem clínica, falsos negativos são mais críticos:')
print(f'  um paciente com arritmia não detectada representa risco real.')

---

## Considerações finais

Este módulo demonstra que uma rede neural MLP simples consegue classificar  
batimentos cardíacos com alta acurácia a partir do sinal bruto de ECG.

### Conexão com as fases seguintes

- **Fase 3 (IoT):** sensores wearables podem capturar sinais de ECG em tempo real  
  e enviar para um modelo como este via MQTT
- **Fase 4 (Visão Computacional):** as imagens de ECG da Fase 1 podem ser processadas  
  com CNNs, complementando a abordagem de sinal deste módulo
- **Fase 6 (Séries Temporais):** os 187 pontos de cada batimento são uma série temporal  
  que pode ser modelada com LSTM ou modelos de atenção

### Limitações

- O modelo foi treinado com dados de 1998 (MIT-BIH) e pode não generalizar para  
  dispositivos modernos com diferentes frequências de amostragem
- A classificação binária simplifica um problema clínico complexo
- Não há análise de viés por gênero neste módulo — o dataset MIT-BIH não documenta  
  o sexo dos pacientes, o que é em si uma limitação de governança

**Este notebook não produz diagnósticos médicos. Tem finalidade exclusivamente acadêmica.**